Sessions 1 & 2 — Project Tasks - P4A 2026

## Create a staging area for analytics use:

```sql
CREATE TABLE a20254350.postings_with_benefits AS
SELECT 
    post.*,
    ben.job_benefits AS benefits
FROM linkedin_jobs.postings post
LEFT JOIN (
    SELECT 
        ben.job_id, 
        GROUP_CONCAT(ben.type SEPARATOR '; ') AS job_benefits
    FROM linkedin_jobs.jobs_benefits ben
    GROUP BY ben.job_id
) ben 
    ON post.job_id = ben.job_id
WHERE post.closed_time IS NULL;
```

> We used `CREATE TABLE ... AS SELECT` because the staging table did not exist yet. Therefore, this approach creates the table and fills it with the selected active postings and concatenated benefits in one cleaner step.

Text-to-sql prompt

You are an AI assistant that converts natural language queries into SQL.

Database context:
The schema is called linkedin_jobs. The original Kaggle dataset description says there are 123,849 LinkedIn job postings from 2023–2024, covering salaries, locations, experience levels, remote work, 
skills, benefits, and company data. However, the actual MySQL table linkedin_jobs.postings available in this project contains 40,059 postings. For SQL analysis, use the rows actually present in 
the database as the source of truth.

Current postings counts:
- Total rows in linkedin_jobs.postings: 40,059
- Active postings where closed_time IS NULL: 39,361
- Closed postings where closed_time IS NOT NULL: 698

MySQL database schema:
Tables:
- companies_companies
- companies_companies_industry
- companies_companies_specialities
- companies_employee_counts
- jobs_benefits
- jobs_job_industries
- jobs_job_skills
- jobs_salaries
- mappings_industries
- mappings_skills
- postings
- salary_summaries_pr

Important rules:
1. Always write MySQL-compatible SQL.
2. Use the schema name linkedin_jobs when referencing tables.
3. The main table for job-posting questions is linkedin_jobs.postings.
4. The database does not have formal primary keys or foreign keys defined.
5. Use inferred joins based on matching ID columns.
6. Prefer LEFT JOIN when starting from postings so job postings are not accidentally removed.
7. Use INNER JOIN only when the user specifically asks for data that must exist in another table, such as skills, benefits, salaries, or industries.
8. Use readable table aliases.
9. Include LIMIT 10 for ranking or “top” questions unless the user requests otherwise.
10. Return only the SQL query unless an explanation is requested.

Main inferred joins:
- postings.company_id = companies_companies.company_id
- postings.job_id = jobs_salaries.job_id
- postings.job_id = jobs_benefits.job_id
- postings.job_id = jobs_job_skills.job_id
- postings.job_id = jobs_job_industries.job_id
- jobs_job_skills.skill_abr = mappings_skills.skill_abr
- jobs_job_industries.industry_id = mappings_industries.industry_id
- companies_companies.company_id = companies_employee_counts.company_id
- companies_companies.company_id = companies_companies_industry.company_id
- companies_companies.company_id = companies_companies_specialities.company_id

Known limitations:
The original dataset description mentions 123,849 postings, but the actual linkedin_jobs.postings table contains 40,059 rows. Use 40,059 as the database count for analysis unless specifically discussing 
the original Kaggle dataset description.

The database tables are not perfectly consistent. Some detail tables contain `job_id` values that are not present in `linkedin_jobs.postings`.

Distinct `job_id` counts found:
- `linkedin_jobs.postings`: 40,059
- `linkedin_jobs.jobs_salaries`: 40,785
- `linkedin_jobs.jobs_benefits`: 30,023
- `linkedin_jobs.jobs_job_skills`: 126,807
- `linkedin_jobs.jobs_job_industries`: 127,125

Because of this, `linkedin_jobs.postings` should be used as the main reference table for job-posting analysis. Joins to detail tables such as `jobs_salaries`, `jobs_benefits`, `jobs_job_skills`, and `jobs_job_industries` 
should be handled carefully, because using `INNER JOIN` may exclude many postings from the result. `LEFT JOIN` is safer when the goal is to keep all postings.

The mapping tables for skills and industries match 100%, so these joins are reliable:
- jobs_job_skills.skill_abr = mappings_skills.skill_abr
- jobs_job_industries.industry_id = mappings_industries.industry_id
   
Use the uploaded `linkedin_jobs table info.csv` file as the reference for exact table and column names.   
    

## Example 1: Question: Which companies have the most job postings?

```sql
SELECT 
    COALESCE(c.name, p.company_name) AS company_name,
    COUNT(*) AS job_count
FROM linkedin_jobs.postings p
LEFT JOIN linkedin_jobs.companies_companies c
    ON p.company_id = c.company_id
GROUP BY COALESCE(c.name, p.company_name)
ORDER BY job_count DESC
LIMIT 10;
```

## Example 2: Question: What are the most common skills for Data Analyst jobs?

```sql
SELECT 
    ms.skill_name,
    COUNT(DISTINCT p.job_id) AS job_count
FROM linkedin_jobs.postings p
INNER JOIN linkedin_jobs.jobs_job_skills jsk
    ON p.job_id = jsk.job_id
INNER JOIN linkedin_jobs.mappings_skills ms
    ON jsk.skill_abr = ms.skill_abr
WHERE p.title LIKE '%Data Analyst%'
GROUP BY ms.skill_name
ORDER BY job_count DESC
LIMIT 10;
```

## Example 3: Question: What is the average normalized salary by experience level?

```sql
SELECT 
    p.formatted_experience_level,
    COUNT(*) AS job_count,
    ROUND(AVG(p.normalized_salary), 2) AS avg_normalized_salary
FROM linkedin_jobs.postings p
WHERE p.normalized_salary IS NOT NULL
  AND p.formatted_experience_level IS NOT NULL
  AND TRIM(p.formatted_experience_level) <> ''
GROUP BY p.formatted_experience_level
ORDER BY avg_normalized_salary DESC;
```

Discussion of results: 

a) Accuracy of the generated SQL

As for the level of accuracy of the generated queries, we can assert they were mostly accurate for questions involving the `postings` table and company information. 
The `postings` table is the safest base table because it contains the main job posting records used in this database. However, queries involving skills, benefits, salaries, or industries 
were less complete because those detail tables do not fully match the postings table. Some detail tables contain job_id values that are not present in postings, and some postings 
do not have matching detail records. As a result, INNER JOIN queries using those tables may exclude many postings.

b) Ambiguities or limitations 

1.	This database does not have defined formal primary keys or foreign keys. Because of this, relationships had to be inferred from column names such as `job_id`, `company_id`, `skill_abr`, and `industry_id`.
2.	Though the original dataset description mentions 123,849 job postings, the actual `linkedin_jobs.postings` table contains 40,059 rows. Therefore, the analysis is based on the records available in the MySQL database.
3.	The salary, benefits, skills, and industry tables do not always match perfectly with the postings table. Some of these tables include job_id values that are not found in postings. Because of this, using INNER JOIN with these tables can remove many job postings from the final results.

c) Possible prompt improvements

The prompt worked well overall, but it could be improved by adding a few sample rows from the main tables. This would make it easier to understand how the data is actually stored and how filters should be written.

It would also help to explain some important fields more clearly, such as normalized_salary, remote_allowed, formatted_work_type, and formatted_experience_level. For example, knowing whether remote_allowed uses 0/1 values or text values would make the generated SQL more accurate.

